# [실습 1] 서울 열린데이터 광장 Open API 활용
- https://data.seoul.go.kr/
- 서울시 공공자전거 실시간 대여정보

In [1]:
# 라이브러리 불러오기
import urllib.request
import json
import pandas as pd

## 1.1. 서울시 공공자전거 실시간 대여정보 1,000건 가져오기

- 서울특별시 공공자전거 실시간 대여정보를 가져와 **bike** 데이터프레임을 만듭니다.
- 데이터셋은 대여소별 실시간 자전거 대여가능 건수, 거치율, 대여소 위치정보를 제공합니다.
- 호출시 시스템 부하로 한번에 최대 **1,000건**를 초과할수 없습니다.
- 우선 1,000개 행만 조회하세요.

In [9]:
# 인증키와  URL 주소
# https://data.seoul.go.kr/dataList/OA-15493/A/1/datasetView.do
# uv add python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()

key = os.getenv('bike_key')
res_type = 'json'
start = 1
end = 1000

url = f'http://openapi.seoul.go.kr:8088/{key}/{res_type}/bikeList/{start}/{end}'

print(url)


http://openapi.seoul.go.kr:8088/52795054456a657838324e4b62547a/json/bikeList/1/1000


In [10]:
response = urllib.request.urlopen(url)
json_str = response.read().decode('utf-8')

print(json_str)

{"rentBikeStatus":{"list_total_count":1000,"RESULT":{"CODE":"INFO-000","MESSAGE":"정상 처리되었습니다."},"row":[{"rackTotCnt":"15","stationName":"102. 망원역 1번출구 앞","parkingBikeTotCnt":"29","shared":"193","stationLatitude":"37.55564880","stationLongitude":"126.91062927","stationId":"ST-4"},{"rackTotCnt":"14","stationName":"103. 망원역 2번출구 앞","parkingBikeTotCnt":"16","shared":"114","stationLatitude":"37.55495071","stationLongitude":"126.91083527","stationId":"ST-5"},{"rackTotCnt":"13","stationName":"104. 합정역 1번출구 앞","parkingBikeTotCnt":"17","shared":"131","stationLatitude":"37.55073929","stationLongitude":"126.91508484","stationId":"ST-6"},{"rackTotCnt":"5","stationName":"105. 합정역 5번출구 앞","parkingBikeTotCnt":"2","shared":"40","stationLatitude":"37.55000687","stationLongitude":"126.91482544","stationId":"ST-7"},{"rackTotCnt":"12","stationName":"106. 합정역 7번출구 앞","parkingBikeTotCnt":"15","shared":"125","stationLatitude":"37.54864502","stationLongitude":"126.91282654","stationId":"ST-8"},{"rackTotCnt":"

**데이터셋 정보**

- rackTotCnt: 거치대개수
- stationName: 대여소이름
- parkingBikeTotCnt: 자전거주차총건수
- shared: 거치율
- stationLatitude: 위도
- stationLongitude: 경도
- stationId: 대여소ID

## 1.2. 서울시 공공자전거 실시간 대여정보 모든 데이터 가져오기

- 위 실습 결과로 작성된 구문을 수정하여 모든 데이터를 가져와 **bike** 데이터프레임을 다시 만듭니다.
- while 문을 사용해 JSON 문자열을 연결 한 후 이후 과정을 진행합니다.

In [53]:
start = 1
end = 1000
result = []
while True:

  start += 1000
  end += 1000

  url = f'http://openapi.seoul.go.kr:8088/{key}/{res_type}/bikeList/{start}/{end}'
  response = urllib.request.urlopen(url)
  json_str = response.read().decode('utf-8')
  # print(json_str)
  json_object = json.loads(json_str)
  bike = pd.json_normalize(json_object['rentBikeStatus']['row'])
  result.append(bike)

  # print(json_object['rentBikeStatus']['list_total_count'])
  if int(json_object['rentBikeStatus']['list_total_count']) < 1000:
    break
print(result.info())

AttributeError: 'list' object has no attribute 'info'

- 하위 데이터를 확인합니다.

In [45]:
json_object = json.loads(json_str)
print(json_object)

# result = json_object['rentBikeStatus']['row']

# 데이터 플레임으로 변환
bike = pd.json_normalize(json_object['rentBikeStatus']['row'])
bike

{'rentBikeStatus': {'list_total_count': 1000, 'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다.'}, 'row': [{'rackTotCnt': '7', 'stationName': '1520. 수유중앙시장', 'parkingBikeTotCnt': '9', 'shared': '129', 'stationLatitude': '37.63986206', 'stationLongitude': '127.02278900', 'stationId': 'ST-650'}, {'rackTotCnt': '10', 'stationName': '1525. 미아동 복합청사', 'parkingBikeTotCnt': '3', 'shared': '30', 'stationLatitude': '37.62714005', 'stationLongitude': '127.02669525', 'stationId': 'ST-647'}, {'rackTotCnt': '10', 'stationName': '1527. 미아역 1번 출구 뒤', 'parkingBikeTotCnt': '8', 'shared': '80', 'stationLatitude': '37.62733459', 'stationLongitude': '127.02605438', 'stationId': 'ST-1101'}, {'rackTotCnt': '10', 'stationName': '1528. 삼각산동 주민센터', 'parkingBikeTotCnt': '2', 'shared': '20', 'stationLatitude': '37.61503601', 'stationLongitude': '127.02069855', 'stationId': 'ST-1102'}, {'rackTotCnt': '15', 'stationName': '1529. 미아동 한국전력공사', 'parkingBikeTotCnt': '31', 'shared': '207', 'stationLatitude': '37.6

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,7,1520. 수유중앙시장,9,129,37.63986206,127.02278900,ST-650
1,10,1525. 미아동 복합청사,3,30,37.62714005,127.02669525,ST-647
2,10,1527. 미아역 1번 출구 뒤,8,80,37.62733459,127.02605438,ST-1101
3,10,1528. 삼각산동 주민센터,2,20,37.61503601,127.02069855,ST-1102
4,15,1529. 미아동 한국전력공사,31,207,37.63018036,127.02490997,ST-1103
...,...,...,...,...,...,...,...
995,10,3907. 신도림7차e편한세상아파트 806동 앞,5,50,37.50862503,126.88353729,ST-2559
996,10,3908. 오류동 다숲오피스텔 앞,21,210,37.49267197,126.84162140,ST-2753
997,10,3910. 목양전원교회 앞 로터리,9,90,37.48389053,126.82218170,ST-2828
998,10,3911. 항동하버리안2단지아파트 후문,5,50,37.48337555,126.82247925,ST-2829


## 1.3. 실시간 대여정보 분석

### 1.3.1.데이터 탐색 및 전처리

- 열 이름, 데이터 형식 등 열 정보를 확인합니다.

In [13]:
bike.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   rackTotCnt         1000 non-null   str  
 1   stationName        1000 non-null   str  
 2   parkingBikeTotCnt  1000 non-null   str  
 3   shared             1000 non-null   str  
 4   stationLatitude    1000 non-null   str  
 5   stationLongitude   1000 non-null   str  
 6   stationId          1000 non-null   str  
dtypes: str(7)
memory usage: 54.8 KB


- 데이터 형식을 int로 변환합니다.
- 대상 열: rackTotCnt, parkingBikeTotCnt, shared
- 구문 예) df['Money'].astype('int')

In [15]:
# 분석을 위해 승하차 인원 컬럼을 숫자형으로 변환
bike[['rackTotCnt', 'parkingBikeTotCnt', 'shared']] = bike[['rackTotCnt', 'parkingBikeTotCnt', 'shared']].astype(int)


- 결과를 확인합니다.

In [16]:

bike.head()

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,29,193,37.55564880,126.91062927,ST-4
1,14,103. 망원역 2번출구 앞,16,114,37.55495071,126.91083527,ST-5
2,13,104. 합정역 1번출구 앞,17,131,37.55073929,126.91508484,ST-6
3,5,105. 합정역 5번출구 앞,2,40,37.55000687,126.91482544,ST-7
4,12,106. 합정역 7번출구 앞,15,125,37.54864502,126.91282654,ST-8


- 기초통계정보를 확인합니다.

In [17]:
bike.describe()

,rackTotCnt,parkingBikeTotCnt,shared
count,1000.000000,1000.00000,1000.000000
mean,13.066000,13.27400,104.510000
std,5.857983,12.93678,95.869896
min,5.000000,0.00000,0.000000
25%,10.000000,4.00000,33.000000
50%,10.000000,10.00000,81.000000
75%,15.000000,19.00000,147.250000
max,51.000000,104.00000,670.000000


### 1.3.2. 가장 많이 거치된 곳 TOP 10

- 자전거가 가장 많이 거치된 곳을 확인하기 위해 parkingBikeTotCnt 열로 내림차순 정렬한 후 상위 10개 행만 출력하세요.

In [18]:
bike_guchi_top10 = bike.sort_values(by='parkingBikeTotCnt',ascending=False).head(10)
bike_guchi_top10

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
645,20,1009. 천호역4번출구(현대백화점),104,520,37.53866577,127.12424469,ST-493
794,30,1210. 롯데월드타워(잠실역2번출구 쪽),85,283,37.51312637,127.10095978,ST-891
87,15,217. NH농협은행 앞,73,487,37.52207947,126.93036652,ST-66
381,15,594. 중랑물재생센터(서울새활용플라자),71,473,37.55830383,127.05805969,ST-967
853,30,1291. 서울체육고등학교 앞,67,223,37.52193832,127.13186646,ST-1580
482,10,762. 오목로 무중력지대 앞,67,670,37.52474213,126.87766266,ST-1503
732,10,1132. 등촌역 7번출구,63,630,37.55110931,126.86370850,ST-528
967,10,1455. 상봉역 2번 출구,63,630,37.59632874,127.08589935,ST-1375
810,13,1231. 잠실역 6번출구,62,477,37.51408768,127.09902954,ST-840
744,36,1152. 마곡역교차로,62,172,37.55831146,126.82652283,ST-1064


### 1.3.3. 거치대 수가 가장 많은 곳 TOP 10

- 거치대 수가 가장 많은 곳을 확인하기 위해 rackTotCnt 열로 내림차순 정렬한 후 상위 10개 행만 출력하세요.

In [19]:
bike_rack_top10 = bike.sort_values(by='rackTotCnt',ascending=False).head(10)
bike_rack_top10

,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
779,51,1193. 마곡센트럴타워 1차,31,61,37.55947113,126.83207703,ST-1688
273,51,450. 효자동 삼거리,16,31,37.58360291,126.97254944,ST-978
83,46,207. 여의나루역 1번출구 앞,1,2,37.52715683,126.93190002,ST-73
533,40,829. 한강대교 북단 교차로,22,55,37.52293015,126.96169281,ST-1326
68,40,186. 월드컵공원,54,135,37.56396484,126.89820862,ST-344
468,40,744. 신목동역 2번 출구,17,43,37.54384232,126.88254547,ST-1015
293,40,474.동대문역사문화공원역 1번출구 뒤편,8,20,37.56629944,127.00849152,ST-1780
247,37,420. 서울시 공공자전거 상암센터,34,92,37.56624603,126.89617920,ST-85
744,36,1152. 마곡역교차로,62,172,37.55831146,126.82652283,ST-1064
18,35,124. 서강대 정문 건너편,41,117,37.55113983,126.93698883,ST-26


### 1.3.4. 지도 시각화

- 위도와 경도 정보가 있으므로 지도상에 거치대 위치를 표시할 수 있습니다.
- 지도 시각화를 위해 folium 라이브러리를 설치합니다.

In [ ]:
# 라이브러리 설치
# %pip install folium

- 위 분석에서 거치대가 가장 많은 곳의 좌표를 지도상에 표시합니다.

In [ ]:
import folium

my_map = folium.Map(location=[],zoom_start=13)